# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading, exploring, and processing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Title:', metadata.name)
print('Version:', metadata.version)
print('\nDescription:')
print(metadata.description)

## 2. Data Overview
Explore available record sets, their fields, and columns as defined in the Croissant schema.

**Note:** We reference all record sets, fields, and columns by their `@id` according to Croissant best practices.

In [ ]:
# List available record sets and their fields by their @id

record_sets = []
if hasattr(metadata, 'record_sets'):
    # mlcroissant >=0.7 uses .record_sets, older versions may use .recordset or .record_sets as a property
    record_sets = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet

print(f'Found {len(record_sets)} record sets:')
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {rs['@id']}  Name: {rs.get('name','(no name)')}")

    if 'fields' in rs:
        for f_idx, field in enumerate(rs['fields']):
            print(f"    Field {f_idx}: @id: {field.get('@id','')}  name: {field.get('name','')}  dataType: {field.get('dataType','')}")
    if 'columns' in rs:
        for c_idx, col in enumerate(rs['columns']):
            print(f"    Column {c_idx}: @id: {col.get('@id','')}  name: {col.get('name','')}  dataType: {col.get('dataType','')}")
    print('---')

# If no record sets found, explain to the user
if not record_sets:
    print("No record sets defined in the Croissant metadata. The dataset may be purely metadata or not structured with record sets in Croissant schema.")

## 3. Data Extraction
Load rows (records) for each record set by `@id` using `mlcroissant`. Each extracted DataFrame follows `@id` conventions for column access.

If no record sets are discovered, skip data extraction.

In [ ]:
# Collect DataFrames by record set @id
dataframes = {}
all_recordset_ids = []

for rs in record_sets:
    rs_id = rs['@id']
    all_recordset_ids.append(rs_id)
    print(f"Loading records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
        else:
            print('  No records found.')
    except Exception as e:
        print(f"  Error loading records for {rs_id}: {str(e)}")
        continue

if dataframes:
    # Show the columns and first rows for the first available record set
    first_rs_id = list(dataframes.keys())[0]
    first_df = dataframes[first_rs_id]
    print(f'\nSample columns for RecordSet {first_rs_id}:')
    print(first_df.columns.tolist())
    first_df.head()
else:
    print('No dataframes available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing: filtering numeric fields, normalization, and grouping. Reference all fields/columns by their Croissant `@id`. If no numeric fields are discovered, adapt accordingly.

In [ ]:
import numpy as np

# Choose a DataFrame with data
if dataframes:
    # Use the first dataframe by recordset @id
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Running EDA on record set: {rs_id}")
    # Find numeric fields (float/int columns)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    if not numeric_candidates:
        # Try to convert object columns to numeric if possible
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c], errors='ignore')
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_candidates:
        # Pick the first numeric field for demonstration
        numeric_field = numeric_candidates[0]
        print(f"Analyzing numeric field: {numeric_field}")

        # Filter (arbitrary threshold = 10 for demo)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first string/categorical field
        group_fields = df.select_dtypes(include=[object, 'category']).columns.tolist()
        group_field = None
        for gf in group_fields:
            if gf != numeric_field:
                group_field = gf
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print('No numeric fields found. Please manually inspect the DataFrame to select a field for analysis.')
else:
    print('No record set DataFrame available for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to the group field (if present).

You can adapt this cell to use your own preferred plotting library.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group (if available)
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
In this notebook, you have:
- Loaded a Croissant-defined FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library
- Explored dataset metadata and record set structure using `@id` references
- Extracted record set data and performed basic exploratory data analysis (filtering, normalization, grouping)
- Visualized key numeric fields to understand distributions and relationships

**Next steps**: Adapt EDA and modeling to your project needs. For more documentation, see the [mlcroissant examples](https://github.com/mlcommons/croissant/tree/main/examples) and [specification](https://mlcommons.github.io/croissant/).